# Pipeline Completo: Focalización de Pobreza Monetaria Urbana ENAHO (2024 - 2025)
## Módulos 01, 02, 03, 05 y 34 — Lima Metropolitana y Callao (`DOMINIO == 8`)

> **Curso:** Inteligencia Artificial (1INF24) — Pontificia Universidad Católica del Perú (PUCP)  
> **Objetivo de este Notebook:** Ejecutar de extremo a extremo la ingesta y fusión relacional de los microdatos ENAHO (2024–2025), el análisis exploratorio y de hogares panel, el preprocesamiento libre de fuga de datos, la validación cruzada espacial en 16 modelos, la evaluación ciega Fuera de Tiempo (OOT 2025), el contraste de hipótesis por *Cluster Bootstrap*, la explicabilidad con TreeSHAP y la auditoría ética por subgrupos.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Configurar acceso a src/
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.io_enaho import (
    load_modulo34,
    load_modulo01,
    load_modulo02,
    load_modulo03,
    load_modulo05,
    KEY_HOGAR,
    KEY_PERSONA,
)

print(f"✓ Entorno configurado correctamente. Raíz del proyecto: {ROOT_DIR}")

---  
## 1. Módulo 34: Sumaria (Resumen de Hogares, Pobreza y Target)

- **Nivel:** Hogar (`conglomerado`, `vivienda`, `hogar`).
- **Filtro:** `dominio == 8` (Lima Metropolitana y Callao).
- **Columnas estandarizadas:** `pobreza_oficial`, `target_pobreza` (1 = Pobre, 0 = No pobre), `factor_expansion`, `miembros_hogar`, `gasto_anual_hogar`, `linea_pobreza`, `lngpc`.

In [ ]:
m34_2024 = load_modulo34(2024, data_root=ROOT_DIR / "Data")
m34_2025 = load_modulo34(2025, data_root=ROOT_DIR / "Data")

print(f"[Módulo 34 - 2024] Hogares: {len(m34_2024):,d} | Pobres: {m34_2024['target_pobreza'].sum():,d} ({m34_2024['target_pobreza'].mean():.2%})")
print(f"[Módulo 34 - 2025] Hogares: {len(m34_2025):,d} | Pobres: {m34_2025['target_pobreza'].sum():,d} ({m34_2025['target_pobreza'].mean():.2%})")
display(m34_2024.head(5))

---  
## 2. Módulo 01: Características de la Vivienda y del Hogar

- **Nivel:** Hogar.
- **Filtro:** `resultado_encuesta in [1, 2]`.
- **Columnas estandarizadas:** `tipo_vivienda`, `material_pared`, `material_piso`, `total_habitaciones`, `total_dormitorios`, `tenencia_vivienda`, `titulo_propiedad`, `registro_sunarp`, `fuente_agua`, `conexion_bano`, `combustible_cocina`, `tiene_internet`, `es_hogar_secundario`.
- **Integridad de Datos Faltantes:** Conservación estricta de valores nulos (`NaN`) en hogares secundarios (`es_hogar_secundario == 1`). **Cero rellenado automático `ffill/bfill`**.

In [ ]:
m01_2024 = load_modulo01(2024, data_root=ROOT_DIR / "Data")
m01_2025 = load_modulo01(2025, data_root=ROOT_DIR / "Data")

# Delimitar a hogares de Lima Metropolitana
hogares_lima_2024 = m34_2024[KEY_HOGAR].drop_duplicates()
hogares_lima_2025 = m34_2025[KEY_HOGAR].drop_duplicates()

m01_2024 = m01_2024.merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m01_2025 = m01_2025.merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 01 - 2024] Registros: {len(m01_2024):,d} | Hogares secundarios: {(m01_2024['es_hogar_secundario'] == 1).sum()}")
print(f"                  Nulos sin imputar en tipo_vivienda:     {m01_2024['tipo_vivienda'].isna().sum()}")
print(f"                  Nulos sin imputar en total_habitaciones: {m01_2024['total_habitaciones'].isna().sum()}")

print(f"[Módulo 01 - 2025] Registros: {len(m01_2025):,d} | Hogares secundarios: {(m01_2025['es_hogar_secundario'] == 1).sum()}")
print(f"                  Nulos sin imputar en tipo_vivienda:     {m01_2025['tipo_vivienda'].isna().sum()}")
print(f"                  Nulos sin imputar en total_habitaciones: {m01_2025['total_habitaciones'].isna().sum()}")
display(m01_2024.head(5))

---  
## 3. Módulo 02: Demografía y Características de los Miembros

- **Nivel:** Persona (`conglomerado`, `vivienda`, `hogar`, `cod_persona`).
- **Filtro:** Residentes habituales (`es_miembro_habitual == 1` y no ausente).
- **Columnas estandarizadas:** `parentesco_jefe`, `sexo` (1: Hombre, 2: Mujer), `edad`, `estado_civil`.

In [ ]:
m02_2024 = load_modulo02(2024, data_root=ROOT_DIR / "Data").merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m02_2025 = load_modulo02(2025, data_root=ROOT_DIR / "Data").merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 02 - 2024] Residentes: {len(m02_2024):,d} en {m02_2024[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
print(f"[Módulo 02 - 2025] Residentes: {len(m02_2025):,d} en {m02_2025[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
display(m02_2024.head(5))

---  
## 4. Módulo 03: Educación y Capital Humano

- **Nivel:** Persona (miembros $\ge 3$ años).
- **Columnas estandarizadas:** `parentesco_jefe`, `lengua_materna`, `nivel_educativo`, `sabe_leer_escribir`, `asiste_centro_educativo`, `tipo_institucion_educ`.

In [ ]:
m03_2024 = load_modulo03(2024, data_root=ROOT_DIR / "Data").merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m03_2025 = load_modulo03(2025, data_root=ROOT_DIR / "Data").merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 03 - 2024] Personas evaluadas: {len(m03_2024):,d} en {m03_2024[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
print(f"[Módulo 03 - 2025] Personas evaluadas: {len(m03_2025):,d} en {m03_2025[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
display(m03_2024.head(5))

---  
## 5. Módulo 05: Empleo e Informalidad Laboral

- **Nivel:** Persona (PET $\ge 14$ años).
- **Columnas estandarizadas:** `condicion_actividad` (1: Ocupado), `categoria_ocupacional`, `empresa_tiene_ruc`, `horas_semanales_trabajadas`, `afiliacion_pension` (5: Sin pensión).
- **Blindaje metodológico:** Exclusión estricta de ingresos laborales/no laborales (cero *data leakage*).

In [ ]:
m05_2024 = load_modulo05(2024, data_root=ROOT_DIR / "Data").merge(hogares_lima_2024, on=KEY_HOGAR, how="inner")
m05_2025 = load_modulo05(2025, data_root=ROOT_DIR / "Data").merge(hogares_lima_2025, on=KEY_HOGAR, how="inner")

print(f"[Módulo 05 - 2024] Personas en edad de trabajar: {len(m05_2024):,d} en {m05_2024[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
print(f"[Módulo 05 - 2025] Personas en edad de trabajar: {len(m05_2025):,d} en {m05_2025[KEY_HOGAR].drop_duplicates().shape[0]:,d} hogares")
display(m05_2024.head(5))

---  
## 6. Matriz Consolidada de Auditoría de Microdatos (2024 vs. 2025)

Cotejo de integridad dimensional y validación de unicidad de llaves primarias (`PK`).

In [ ]:
def audit_year_modules(year: int, mods: dict) -> pd.DataFrame:
    rows = []
    for name, df in mods.items():
        pk = KEY_HOGAR if name in ["modulo34", "modulo01"] else KEY_PERSONA
        rows.append({
            "Año": year,
            "Módulo": name,
            "Nivel": "Hogar" if name in ["modulo34", "modulo01"] else "Persona",
            "Filas Totales": len(df),
            "Hogares Únicos": df[KEY_HOGAR].drop_duplicates().shape[0],
            "Columnas": df.shape[1],
            "PK Única": not df.duplicated(subset=pk).any(),
        })
    return pd.DataFrame(rows)

mods_2024_all = {
    "modulo34": m34_2024,
    "modulo01": m01_2024,
    "modulo02": m02_2024,
    "modulo03": m03_2024,
    "modulo05": m05_2024,
}

mods_2025_all = {
    "modulo34": m34_2025,
    "modulo01": m01_2025,
    "modulo02": m02_2025,
    "modulo03": m03_2025,
    "modulo05": m05_2025,
}

audit_df = pd.concat([audit_year_modules(2024, mods_2024_all), audit_year_modules(2025, mods_2025_all)], ignore_index=True)
display(audit_df)

---
## 7. Consolidación del Big DataFrame Unificado Multi-Año (Operador $\\Phi$)

Integración horizontal de los 5 módulos en una sola tabla tabular a nivel hogar utilizando `src.dataset.build_combined_dataset()`:
- **Reducción relacional:** Miembros colapsados a hogar (rama de jefatura y funcionales agregados de menores, adultos mayores y educación).
- **Fusión horizontal:** Enlace con Módulo 34 y Módulo 01 por la llave primaria `["conglomerado", "vivienda", "hogar"]`.
- **Unión pura sin purgas:** Se concatenan 2024 y 2025 ($N = 8{,}219$ hogares) con la columna `year` para permitir el análisis temporal y la detección de hogares panel en el EDA.
- **Delimitación estricta:** Cero preprocesamiento todavía (sin purga de panel, sin imputación de nulos y sin recorte de outliers).

In [ ]:
from src.dataset import build_combined_dataset

df_all = build_combined_dataset(years=(2024, 2025), data_root=ROOT_DIR / "Data")

print(f"✓ Big DataFrame Unificado (2024 + 2025): {df_all.shape}")
print(df_all.groupby("year")["target_pobreza"].agg(hogares="count", pobres="sum", tasa_pobreza="mean"))
display(df_all.head(5))

In [ ]:
# Inventario de columnas del Big DataFrame consolidado
print(f"Total variables en el Big DF: {len(df_all.columns)}")
print(list(df_all.columns))

---
## 8. Selección Canónica del Espacio de Características ($\\mathbf{x}_i \\in \\mathbb{R}^{28}$)

De las 55 columnas integradas en el Big DataFrame, delimitamos estrictamente las **28 características canónicas** formalizadas en la Sección 3 del informe metodológico:
$$\\mathbf{x}_i = [\\mathbf{x}_i^{(\\text{viv})}, \\mathbf{x}_i^{(\\text{dem})}, \\mathbf{x}_i^{(\\text{edu})}, \\mathbf{x}_i^{(\\text{emp})}] \\in \\mathbb{R}^{28}$$

### Criterios de Exclusión:
1. **Variables administrativas, temporales y de fuga:** Llaves de identificación (`conglomerado`, `vivienda`, `hogar`), geográficas (`ubigeo`, `dominio`, `estrato`), ponderadores (`factor_expansion`), etiquetas oficiales (`pobreza_oficial`, `target_pobreza`, `y`), variables monetarias de fuga (`gasto_anual_hogar`, `linea_pobreza`, `lngpc`) y la columna `year`.
2. **5 indicadores NBI agregados:** `nbi_vivienda_inadecuada`, `nbi_hacinamiento`, `nbi_sin_bano`, `nbi_nino_sin_escuela`, `nbi_alta_dependencia` (cuasi-constantes en Lima urbana y redundantes con las variables físicas directas).
3. **Variables intermedias/redundantes:** `miembros_hogar` (duplicado de `n_miembros`), `jefe_estado_civil`, `es_hogar_secundario`, `sin_datos_vivienda`, etc.

In [ ]:
from src.preprocessing import (
    NUMERIC_COLS,
    CATEGORICAL_COLS,
    CANONICAL_FEATURES_28,
    TARGET_COL,
    GROUP_COL,
    prepare_xy_data,
    SecondaryHouseholdImputer,
    build_preprocessor,
)

print(f"✓ Total características canónicas: {len(CANONICAL_FEATURES_28)}")
print(f"  • Numéricas / Ratios / Ordinales (14): {NUMERIC_COLS}")
print(f"  • Categóricas / Nominales / Binarias (14): {CATEGORICAL_COLS}")

missing_in_df = [c for c in CANONICAL_FEATURES_28 if c not in df_all.columns]
assert len(missing_in_df) == 0, f"Variables faltantes: {missing_in_df}"
print("✓ Todas las 28 variables están presentes en df_all (2024 y 2025).")

---
## 9. Análisis Exploratorio de Datos (EDA): Diagnóstico Temporal (2024 vs 2025) y Train 2024

Primero auditamos el **Big DataFrame unificado (`df_all`, 2024 + 2025)** para cuantificar:
1. **Diferencia temporal interanual (2024 vs. 2025):** Evolución de la pobreza y tamaño muestral.
2. **Detección de hogares panel repetidos:** Identificación de hogares entrevistados en ambos años por `(conglomerado, vivienda, hogar)` que deben purgarse de 2025 antes de modelar.

Luego realizamos el EDA detallado de características exclusivamente sobre **ENAHO 2024 (Train/Val, $N = 4{,}090$)** para evitar fuga de información del conjunto de prueba.

### 9.0. Análisis Temporal Interanual y Detección de Hogares Panel (2024 vs. 2025)

In [ ]:
from src.io_enaho import KEY_HOGAR

df_24_raw = df_all[df_all["year"] == 2024].copy()
df_25_raw = df_all[df_all["year"] == 2025].copy()

# Detección de hogares panel repetidos entre 2024 y 2025
panel_keys = df_24_raw[KEY_HOGAR].merge(df_25_raw[KEY_HOGAR], on=KEY_HOGAR).assign(es_panel=True)
df_25_diag = df_25_raw.merge(panel_keys, on=KEY_HOGAR, how="left")
df_25_diag["es_panel"] = df_25_diag["es_panel"].fillna(False).astype(bool)

n_panel = int(df_25_diag["es_panel"].sum())
n_nuevos = int((~df_25_diag["es_panel"]).sum())
cong_24 = set(df_24_raw["conglomerado"].unique())
cong_25 = set(df_25_raw["conglomerado"].unique())

print("=== DIAGNÓSTICO TEMPORAL Y PANEL (2024 vs. 2025) ===")
print(f"Hogares 2024:                  {len(df_24_raw):,d} | Pobreza: {df_24_raw[TARGET_COL].mean():.2%}")
print(f"Hogares 2025 (bruto):          {len(df_25_raw):,d} | Pobreza: {df_25_raw[TARGET_COL].mean():.2%}")
print(f"  • Hogares panel repetidos:   {n_panel:,d} | Pobreza: {df_25_diag.loc[df_25_diag['es_panel'], TARGET_COL].mean():.2%}")
print(f"  • Hogares nuevos (OOT puro): {n_nuevos:,d} | Pobreza: {df_25_diag.loc[~df_25_diag['es_panel'], TARGET_COL].mean():.2%}")
print(f"Conglomerados 2024: {len(cong_24)} | 2025: {len(cong_25)} | Compartidos: {len(cong_24 & cong_25)}")

# Definimos df_2024 para el EDA de variables sin tocar 2025
df_2024 = df_24_raw.reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 10, "figure.autolayout": True})

# 9.1. Balance de la variable objetivo
n_total = len(df_2024)
n_poor = int(df_2024[TARGET_COL].sum())
n_non_poor = n_total - n_poor
p_poor = n_poor / n_total
gamma = n_non_poor / n_poor

print("=== DISTRIBUCIÓN DE CLASES (ENAHO 2024) ===")
print(f"Total hogares:       {n_total:,d}")
print(f"Hogares no pobres:   {n_non_poor:,d} ({1 - p_poor:.2%})")
print(f"Hogares pobres (y=1):{n_poor:,d} ({p_poor:.2%})")
print(f"Razón de desbalance: γ ≈ {gamma:.2f} : 1")

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.countplot(x=TARGET_COL, data=df_2024, ax=ax, hue=TARGET_COL, palette=["#2b5c8f", "#d95f02"], legend=False)
ax.set_title(f"Balance de Clases en ENAHO 2024 (Pobreza = {p_poor:.2%}, γ = {gamma:.2f})")
ax.set_xlabel("Target Pobreza Monetaria (0 = No Pobre, 1 = Pobre)")
ax.set_ylabel("Número de Hogares")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,d} ({p.get_height()/n_total:.1%})",
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha="center", va="center", color="white", fontweight="bold")
plt.show()

### 9.2. Auditoría Exhaustiva de Valores Faltantes (Nulos)

Evaluamos los valores faltantes en las 28 variables canónicas antes de cualquier imputación:
- **72 hogares secundarios (1.76%):** Carecen de datos en Módulo 01 (`tipo_vivienda`, `material_pared`, `material_piso`, `total_habitaciones`, `total_dormitorios`) por compartir infraestructura con el hogar principal.
- **1,232 jefes inactivos/desempleados:** Tienen nulos en `jefe_horas_semanales`, `jefe_cat_ocupacional` y `jefe_empresa_ruc` porque no trabajan (nulo estructural legítimo).
- **225 hogares sin red de agua continua:** NA en `agua_todos_los_dias`.
- **46 hogares que no cocinan con combustible tradicional:** NA en `combustible_cocina`.

In [ ]:
null_summary = df_2024[CANONICAL_FEATURES_28].isnull().sum()
null_df = pd.DataFrame({
    "Variable": null_summary.index,
    "Nulos": null_summary.values,
    "Porcentaje_Nulos": (null_summary.values / len(df_2024)) * 100
})
null_df = null_df[null_df["Nulos"] > 0].sort_values(by="Nulos", ascending=False).reset_index(drop=True)
display(null_df)

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.barplot(data=null_df, x="Porcentaje_Nulos", y="Variable", ax=ax, hue="Variable", palette="mako", legend=False)
ax.set_title("Porcentaje de Valores Faltantes por Variable (ENAHO 2024)")
ax.set_xlabel("% de Nulos")
ax.set_ylabel("Característica")
plt.show()

### 9.3. Estadísticos Descriptivos y Preservación de Valores Atípicos

De acuerdo con la metodología del proyecto y la recomendación de *Grosh et al. (2022)*, **no se realiza recorte ni poda de valores atípicos** (*outliers*) en tamaño de hogar (`n_miembros`), hacinamiento (`personas_por_hab`) ni tasa de dependencia (`tasa_dependencia`). Podar estos extremos excluiría artificialmente a los hogares más vulnerables y desfavorecidos de Lima Metropolitana. La robustez frente a colas pesadas se garantiza mediante:
1. Invarianza monotónica al orden en clasificadores basados en árboles (CART, Random Forest, LightGBM).
2. Escalamiento robusto basado en mediana y rango intercuartílico (`RobustScaler`) en modelos lineales y PMT-MCO.

In [ ]:
# Estadísticos descriptivos de las variables numéricas
desc = df_2024[NUMERIC_COLS].describe().T[["count", "mean", "std", "min", "50%", "max"]]
desc.columns = ["Conteo", "Media", "Desv_Est", "Min", "Mediana", "Max"]
display(desc.round(2))

# Visualización de distribuciones y outliers clave
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
key_outlier_cols = ["n_miembros", "personas_por_hab", "tasa_dependencia", "jefe_horas_semanales"]
titles = ["Tamaño del Hogar", "Personas por Hab.", "Tasa Dependencia", "Horas Semanales Jefe"]

for ax, col, title in zip(axes, key_outlier_cols, titles):
    sns.boxplot(y=col, x=TARGET_COL, data=df_2024, ax=ax, hue=TARGET_COL, palette=["#2b5c8f", "#d95f02"], legend=False)
    ax.set_title(title)
    ax.set_xlabel("Pobreza (0=No, 1=Sí)")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 9.4. Correlaciones Lineales con la Pobreza Monetaria

Analizamos la matriz de correlación de Pearson entre las características numéricas y el indicador de pobreza $y_i$.
- **Factores de Riesgo / Vulnerabilidad (+):** `personas_por_hab` (+0.34), `n_menores_14` (+0.33), `n_miembros` (+0.33) y `n_ninos_0_5` (+0.29). A mayor carga infantil y hacinamiento, significativamente mayor probabilidad de pobreza.
- **Factores Protectores (-):** `ocupados_por_miembro` (-0.18), `educ_jefe` (-0.17) y `total_habitaciones` (-0.11). La diversificación laboral y el capital humano del jefe actúan como barreras clave ante la vulnerabilidad económica.

In [ ]:
# Matriz de correlación de Pearson
corr_matrix = df_2024[NUMERIC_COLS + [TARGET_COL]].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax
)
ax.set_title("Matriz de Correlación Lineal (Pearson) - ENAHO 2024")
plt.show()

# Ranking de correlación con target_pobreza
print("=== CORRELACIÓN CON TARGET_POBREZA ===")
print(corr_matrix[TARGET_COL].sort_values(ascending=False).to_string())

### 9.5. Categorías Raras y Tasa de Pobreza Condicional

Para evitar la explosión dimensional y el sobreajuste (*overfitting*) en categorías con escasa representatividad (< 1% de la muestra, es decir, < 40 observaciones), el preprocesador agrupará categorías infrecuentes mediante el hiperparámetro `min_frequency=20` en `OneHotEncoder`.

In [ ]:
rare_summary = []
for c in CATEGORICAL_COLS:
    vc = df_2024[c].value_counts(dropna=False)
    n_rare = (vc < 40).sum()
    pct_rare = vc[vc < 40].sum() / len(df_2024) * 100
    rare_summary.append({
        "Variable": c,
        "Categorías_Totales": len(vc),
        "Categorías_<40_obs": n_rare,
        "%_Muestra_en_Raras": round(pct_rare, 2)
    })

rare_df = pd.DataFrame(rare_summary).sort_values(by="Categorías_<40_obs", ascending=False).reset_index(drop=True)
display(rare_df)

---
## 10. Partición OOT (Purga de Panel), Limpieza Anti-Fuga y Verificación de Nulos Cero

### 10.1. Partición Temporal y Política de No Eliminación de Hogares (*Zero Drop-Row Policy*)
Antes de preprocesar, aplicamos `split_oot_train_test(df_all)` para separar `year == 2024` ($N = 4{,}090$) y `year == 2025` purgado de los 930 hogares panel ($N_{\\text{test}} = 3{,}199$).

En encuestas de hogares orientadas a políticas sociales, **descartar filas con nulos (`dropna`) está metodológicamente prohibido**:
1. Los 72 hogares secundarios de 2024 presentan una tasa de pobreza del **27.8%** (frente al 18.9% global). Descartarlos sesgaría el clasificador e invisibilizaría la pobreza en inquilinatos multifamiliares y quintas.
2. Los 225 hogares sin red de agua continua presentan una tasa de pobreza de casi el **40%**.
3. Por ello, se preserva el **100% de las observaciones** ($N_{\\text{train}} = 4{,}090$, $N_{\\text{test}} = 3{,}199$) aplicando una estrategia de imputación estructurada en dos fases.

### 10.2. Protocolo de Imputación en Dos Fases:
- **Fase 1 (`SecondaryHouseholdImputer`):** Imputa habitaciones y dormitorios mediante la **mediana del conglomerado censal** ajustada estrictamente sobre el conjunto de entrenamiento, recalcula el hacinamiento (`personas_por_hab`) de manera relacional, y fija en `0.0` las horas de jefes inactivos.
- **Fase 2 (`ColumnTransformer`):** Imputa con la mediana de entrenamiento los nulos restantes en numéricas (`jefe_edad`, `educ_jefe`, `jefe_horas_semanales`), y asigna el código sentinela `-1.0` (*"desconocido"*) a las categóricas antes de aplicar la codificación One-Hot con poda de raras (< 1%).

In [ ]:
from src.dataset import split_oot_train_test

# 0. Partición Train 2024 / Test OOT 2025 (purgando los 930 hogares panel)
df_2024, df_2025_test = split_oot_train_test(df_all, train_year=[2024], test_year=[2025], purge_panel=True)

from src.preprocessing import (
    build_preprocessor,
    SecondaryHouseholdImputer,
    prepare_xy_data,
    CANONICAL_FEATURES_28,
)

# 1. Extracción de variables canónicas (X, y, groups)
X_tr, y_tr, grp_tr = prepare_xy_data(df_2024)
X_te, y_te, grp_te = prepare_xy_data(df_2025_test)

# Conteo inicial de nulos en las 28 variables
nulls_tr_before = X_tr[CANONICAL_FEATURES_28].isnull().sum()
nulls_te_before = X_te[CANONICAL_FEATURES_28].isnull().sum()

# 2. Fase 1: Limpiador de hogares secundarios y estructura física
cleaner = SecondaryHouseholdImputer(group_col="conglomerado")
X_tr_clean = cleaner.fit_transform(X_tr)
X_te_clean = cleaner.transform(X_te)

nulls_tr_f1 = X_tr_clean[CANONICAL_FEATURES_28].isnull().sum()
nulls_te_f1 = X_te_clean[CANONICAL_FEATURES_28].isnull().sum()

# 3. Fase 2: ColumnTransformer (Imputación final, escalado y One-Hot Encoding)
preprocessor = build_preprocessor(scale_numeric=True, one_hot=True, min_category_frequency=0.01)
X_tr_proc = preprocessor.fit_transform(X_tr_clean)
X_te_proc = preprocessor.transform(X_te_clean)

# Tabla de auditoría del Data Cleaning: Antes vs. Después
vars_with_nulls = nulls_tr_before[nulls_tr_before > 0].index
cleaning_audit = pd.DataFrame({
    "Variable": vars_with_nulls,
    "Nulos_Train_Antes": nulls_tr_before[vars_with_nulls].values,
    "Nulos_Train_Fase1": nulls_tr_f1[vars_with_nulls].values,
    "Nulos_Test_Antes": nulls_te_before[vars_with_nulls].values,
    "Nulos_Test_Fase1": nulls_te_f1[vars_with_nulls].values,
    "Estrategia_Imputación": [
        "Mediana Conglomerado Censal",
        "Mediana Conglomerado Censal",
        "Recálculo Relacional: miembros / hab",
        "Mediana Train (54 años)",
        "Mediana Train (6 = Sec. completa)",
        "Mediana Train / 0.0 jefes inactivos",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('sin red pública continua')",
        "Sentinela -1.0 ('no cocina / desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')",
        "Sentinela -1.0 ('desconocido')"
    ]
})

print("=== TABLA DE AUDITORÍA DEL DATA CLEANING (ANTES VS. DESPUÉS) ===")
display(cleaning_audit)

print("=== BALANCE FINAL DE INTEGRIDAD DE DATOS ===")
print(f"✓ Filas preservadas en Train 2024: {len(X_tr_proc):,d} (0 eliminadas)")
print(f"✓ Filas preservadas en Test 2025:  {len(X_te_proc):,d} (0 eliminadas)")
print(f"✓ Nulos finales en Train 2024:      {np.isnan(X_tr_proc).sum()} ({np.isnan(X_tr_proc).sum() == 0})")
print(f"✓ Nulos finales en Test 2025:       {np.isnan(X_te_proc).sum()} ({np.isnan(X_te_proc).sum() == 0})")
print(f"✓ Dimensión final de la matriz:     {X_tr_proc.shape[1]} columnas densas tras One-Hot Encoding")

---
## 11. Modelado Supervisado Extendido y Validación Cruzada Espacial

Evaluamos bajo `GroupKFold(n_splits=5)` agrupado por `conglomerado` censal sobre ENAHO 2024 ($N = 4{,}090$ hogares):
1. **Comparativa interna de escaladores (`RobustScaler` vs. `StandardScaler` vs. `MinMaxScaler`):** Verificando empíricamente que `RobustScaler` protege a los modelos sensibles a escala frente a los valores extremos preservados en la distribución de hogares.
2. **Benchmark de 16 Modelos en 8 Familias:** Baselines de solo vivienda ($H_1$), econométricos continuos (`PMT_MCO`, `PMT_Quantile_20`), lineales/generativos (`Logistica_ElasticNet`, `LDA`, `Naive_Bayes`), árbol podado (`CART_Podado`), ensambles *bagging* (`Random_Forest`, `Extra_Trees`), *gradient boosting* (`LightGBM_GBDT`, `AdaBoost`), no lineales (`SVM_RBF`, `KNN`, `MLP_Red_Neuronal`) y meta-ensamble (`Stacking_Ensemble`), optimizando hiperparámetros y el peso de costo $\alpha \in \{1, 2, \gamma\}$ según **PR-AUC**.

In [ ]:
from src.models import compare_scalers_cv, run_spatial_cv_benchmark

print("=== 11.1. COMPARATIVA DE ESCALADORES DENTRO DEL K-FOLD (ENAHO 2024) ===")
scaler_comp_df = compare_scalers_cv(df_2024, n_splits=5)
display(scaler_comp_df)

In [ ]:
print("=== 11.2. BENCHMARK COMPLETO DE LAS 16 FAMILIAS / MODELOS EN GROUPKFOLD(5) ===")
best_models_cv_df, all_configs_cv_df, fitted_artifacts_2024 = run_spatial_cv_benchmark(df_2024, n_splits=5)
display(best_models_cv_df)

### 11.3. Boxplots Comparativos con Validación Cruzada Espacial Repetida ($6 \times 5 = 30$ Corridas) y Selección del Mejor Modelo

Para obtener una comparación estadística de alta potencia (en lugar de solo 5 puntos), ejecutamos `run_repeated_spatial_cv()` con **6 repeticiones $\times$ 5 pliegues = 30 corridas espaciales independientes por modelo** (barajando los 704 conglomerados sin partir ninguno entre train y val). Evaluamos Mediana, IQR, Rango Promedio de Friedman y test de Wilcoxon pareado frente al modelo #1.

In [ ]:
from src.models import run_repeated_spatial_cv, rank_models_by_folds, plot_cv_boxplots

repeated_folds_df = run_repeated_spatial_cv(fitted_artifacts_2024, df_2024, n_splits=5, n_repeats=6)
ranking_cv_df = rank_models_by_folds(repeated_folds_df, best_models_cv_df)

print("=== RANKING ESTADÍSTICO EN 30 CORRIDAS ESPACIALES (MEDIANA, IQR, FRIEDMAN Y WILCOXON) ===")
display(ranking_cv_df[[
    "Modelo", "Familia", "N_Corridas", "PR_AUC_Mediana", "PR_AUC_IQR",
    "PR_AUC_Mean_Runs", "PR_AUC_Std_Runs", "Rank_Medio_PR", "Veces_Top1_PR", "Veces_Top3_PR",
    "Wilcoxon_p_vs_Top1", "EE_B20_Mediana", "EE_B20_IQR", "Prec_at_Rec70_OOF"
]])

_ = plot_cv_boxplots(
    repeated_folds_df,
    save_path=ROOT_DIR / "Documentation" / "latex" / "figures" / "cv2024_boxplot_models.png",
    show=True,
)

## 12. Evaluación Fuera de Tiempo (OOT 2025), Inferencia Bootstrap, TreeSHAP y Equidad

En esta sección evaluamos de forma **estrictamente ciega** todos los modelos entrenados y calibrados en ENAHO 2024 sobre los **3,199 hogares independientes de ENAHO 2025** (601 conglomerados, sin hogares panel). Contrastamos formalmente las hipótesis $H_1$–$H_4$ mediante **Cluster Bootstrap** (1,000 réplicas a nivel de conglomerado), explicamos las predicciones con **TreeSHAP** y auditamos sesgos por subgrupos sensibles.

In [ ]:
from src.preprocessing import TARGET_COL
from src.evaluation import (
    evaluate_oot_2025,
    compute_ee_ie_curves,
    plot_ee_ie_curves,
)

oot_2025_df, probs_2025 = evaluate_oot_2025(fitted_artifacts_2024, df_2025_test, budget_coverage=0.20)
print("=== 12.1 EVALUACIÓN CIEGA FUERA DE TIEMPO (ENAHO 2025, N = 3,199) ===")
display(oot_2025_df[[
    "Modelo", "Familia", "PR_AUC_2024_CV", "PR_AUC_2025", "PR_AUC_2025_Pond",
    "Delta_Temp_PR (H4)", "ROC_AUC_2025", "EE_B20_2025", "IE_B20_2025",
    "EE_B20_2025_Pond", "Recall_Tau_2025 (H3)", "Prec_Tau_2025 (H3)", "Brier_2025"
]])

curves_2025_df = compute_ee_ie_curves(df_2025_test[TARGET_COL].values, probs_2025)
_ = plot_ee_ie_curves(
    curves_2025_df,
    save_path=ROOT_DIR / "Documentation" / "latex" / "figures" / "ee_ie_curves_2025.png",
    show=True,
)

In [ ]:
from src.evaluation import run_cluster_bootstrap_hypotheses

hyp_bootstrap_df = run_cluster_bootstrap_hypotheses(
    df_2024, df_2025_test, fitted_artifacts_2024, probs_2025, n_boot=1000, seed=42
)
print("=== 12.2 CONTRASTE FORMAL DE HIPÓTESIS H1-H4 (CLUSTER BOOTSTRAP, 1,000 RÉPLICAS) ===")
display(hyp_bootstrap_df)

In [ ]:
from src.evaluation import compute_treeshap_analysis, audit_subgroup_fairness_2025

shap_res_2025 = compute_treeshap_analysis(
    fitted_artifacts_2024["LightGBM_GBDT"],
    df_2025_test,
    save_dir=ROOT_DIR / "Documentation" / "latex" / "figures",
    show=True,
)

print("=== 12.3 EXPLICABILIDAD TREESHAP — TOP 15 VARIABLES CANÓNICAS (R^28) ===")
display(shap_res_2025["original_importance"].head(15))

print("=== 12.3B CONTRIBUCIÓN GLOBAL POR BLOQUE TEMÁTICO ENAHO ===")
display(shap_res_2025["block_importance"])

fairness_2025_df = audit_subgroup_fairness_2025(
    df_2025_test, probs_2025, fitted_artifacts_2024, model_name="LightGBM_GBDT"
)
print("=== 12.4 AUDITORÍA ÉTICA DE EQUIDAD POR SUBGRUPOS (TEST OOT 2025) ===")
display(fairness_2025_df)

## 13. Aplicación del Modelo Ganador: Exportación Anti-Overfitting y Prueba en Encuesta de Hogar

Habiendo demostrado que **`Stacking_Ensemble`** encabeza la evaluación ciega Fuera de Tiempo (OOT 2025), esta sección final consolida el modelo definitivo de producción integrando las campañas **ENAHO 2024 + 2025** bajo cuatro salvaguardas formales contra el sobreajuste (*overfitting*):
1. **Deduplicación longitudinal del panel (`conlome`, `vivienda`, `hogar`):** se depuran los `930` hogares repetidos entre 2024 y 2025 conservando su registro más reciente (`N = 7,289` hogares físicos únicos en `706` conglomerados), evitando que un mismo hogar reciba doble peso o memorización.
2. **Regularización estructural acotada (`Stacking_Tight`):** se fijan hiperparámetros conservadores (`ElasticNet C=0.15`, `RandomForest depth=7 / min_leaf=30`, `ExtraTrees depth=8 / min_leaf=30`, `HistGBDT max_iter=120 / lr=0.03 / leaf=10 / depth=4 / l2=8.0` y meta-modelo `LogisticRegression C=0.2`), reduciendo la brecha Train vs. OOF (`Gap_Overfit_PR`) por debajo de `0.05`.
3. **Calibración de umbrales estrictamente Out-Of-Fold (`GroupKFold(5)` por conglomerado):** tanto $\tau_{\text{rec70}}$ como el corte presupuestal Top 20% (`cutoff_b20`) y la distribución empírica de percentiles se fijan exclusivamente sobre predicciones fuera de muestra antes de ajustar el artefacto final serializado.
4. **Interfaz de encuesta socioeconómica con explicabilidad local:** `src/inference.py` (y su ejecutable de consola `src/encuesta_pobreza.py`) traduce respuestas humanas en español según el catálogo oficial INEI ENAHO al vector $\mathbb{R}^{28}$, calcula los ratios derivados y descompone cada predicción mediante TreeSHAP exacto.

In [ ]:
from src.inference import (
    DEFAULT_BUNDLE_PATH,
    PRESET_HOUSEHOLDS,
    train_and_export_deployment_bundle,
    load_deployment_bundle,
    predict_single_household,
    format_prediction_report,
    run_interactive_survey,
)

# 13.1 Consolidación multi-año (2024+2025 deduplicado), auditoría Train vs. OOF y exportación .joblib
winner_bundle, anti_overfit_audit_df = train_and_export_deployment_bundle(
    years=(2024, 2025),
    save_path=DEFAULT_BUNDLE_PATH,
    budget_coverage=0.20,
    verbose=True,
)
display(anti_overfit_audit_df)

In [ ]:
# 13.2 Evaluación del artefacto exportado sobre perfiles de referencia y encuesta editable
winner_bundle = load_deployment_bundle(DEFAULT_BUNDLE_PATH)

preset_summary = []
for preset_name, preset_answers in PRESET_HOUSEHOLDS.items():
    res_p = predict_single_household(winner_bundle, preset_answers)
    preset_summary.append({
        "Perfil": preset_name,
        "Descripcion": preset_answers["descripcion"],
        "Prob_Stacking_%": round(res_p["prob_pobreza"] * 100, 2),
        "Prob_HistGBDT_%": round(res_p["prob_gbdt"] * 100, 2),
        "Percentil_Lima": round(res_p["percentil_riesgo_lima"], 1),
        "En_Riesgo_Tau": res_p["en_riesgo_tau"],
        "Califica_Top20": res_p["prioritario_top20"],
    })
print("=== 13.2A COMPARATIVA DE PERFILES DE REFERENCIA ENAHO ===")
display(pd.DataFrame(preset_summary))

# Formulario editable de prueba rápida (códigos oficiales INEI ENAHO M01/M02/M03/M05)
mi_hogar = {
    "es_hogar_secundario": False,
    "tipo_vivienda": 1,          # 1=Casa indep., 2=Departamento, 3=Quinta, 6=Improvisada
    "material_pared": 1,         # 1=Ladrillo/cemento, 3=Adobe, 8=Triplay/calamina/estera
    "material_piso": 5,          # 1=Parquet, 3=Losetas, 5=Cemento, 6=Tierra
    "total_habitaciones": 3,
    "total_dormitorios": 2,
    "tenencia_vivienda": 1,      # 1=Alquilada, 2=Propia pagada, 4=A plazos, 6=Cedida
    "fuente_agua": 1,            # 1=Red pública dentro vivienda, 4=Camión-cisterna
    "agua_todos_los_dias": 1,    # 1=Sí, 0=No
    "conexion_bano": 1,          # 1=Red pública dentro vivienda, 5=Pozo ciego
    "combustible_cocina": 2,     # 2=Gas GLP, 3=Gas natural, 6=Leña, 8=No cocinan
    "tiene_internet": 1,         # 1=Sí, 0=No
    "n_miembros": 4,
    "n_ninos_0_5": 1,
    "n_menores_14": 2,
    "n_mayores_65": 0,
    "jefe_edad": 38,
    "jefe_mujer": 0,             # 0=Hombre, 1=Mujer
    "educ_jefe": 6,              # 1=Sin nivel .. 6=Sec. completa .. 10=Univ. completa
    "educ_max_hogar": 8,         # Nivel máximo entre miembros (>= educ_jefe)
    "n_ocupados": 1,
    "jefe_ocupado": 1,           # 1=Ocupado, 0=Desempleado/Inactivo
    "jefe_horas_semanales": 45.0,
    "jefe_cat_ocupacional": 4,   # 1=Empleador, 2=Indep., 3=Empleado, 4=Obrero
    "jefe_empresa_ruc": 2,       # 1=Pers. Jurídica, 2=Pers. Natural c/RUC, 3=Sin RUC
    "jefe_sin_pension": 1,       # 0=Afiliado AFP/ONP, 1=Sin pensión
}

res_mi_hogar = predict_single_household(winner_bundle, mi_hogar)
print(format_prediction_report(res_mi_hogar, label="Formulario Editable (mi_hogar)"))
display(res_mi_hogar["contribuciones_28"].head(10))

# Cambia a True para responder la encuesta paso a paso directamente en esta celda
# (o ejecuta en terminal: python3 src/encuesta_pobreza.py)
EJECUTAR_ENCUESTA_INTERACTIVA = True
if EJECUTAR_ENCUESTA_INTERACTIVA:
    run_interactive_survey(winner_bundle)